# 00b · Validación del ambiente (instructor)
> **Workshop Tec de Monterrey · De los datos a los agentes**: Abarrotera del Centro S.A. (ficticia).

Corre este notebook **30 minutos antes del evento**, después de `00`–`06` (incluido `03c_call_center`) (y `08` si harás la demo de Telegram). Cada fila dice **PASS** o **FAIL** con el valor encontrado y lo esperado. Si algo falla, la columna `si_falla` dice qué notebook volver a correr.

In [ ]:
%%sql -r ctx
USE ROLE TEC_ADMIN;
USE WAREHOUSE TEC_WH;

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

s = get_active_session()
q = lambda sql: s.sql(sql).collect()[0][0]

checks = [
    # (área, check, sql, esperado, comparación, si_falla)
    ('Datos', 'Archivos en el stage', "SELECT COUNT(*) FROM DIRECTORY(@TEC_DEMO.RAW.DATOS_CSV) WHERE relative_path LIKE '%.csv%'", 26, '==', '01_datos (copia de archivos)'),
    ('Datos', 'Audios del call center en el stage', "SELECT COUNT(*) FROM DIRECTORY(@TEC_DEMO.RAW.AUDIOS) WHERE relative_path LIKE '%.mp3'", 13, '==', '01_datos (copia de audios)'),
    ('Datos', 'Venta de prueba residual (demo datos vivos)', "SELECT COUNT(*) FROM TEC_DEMO.RAW.VENTAS WHERE venta_id = 999999999", 0, '==', '02 (celda de limpieza)'),
    ('Pipeline', 'Dynamic Tables', "SELECT COUNT(*) FROM TEC_DEMO.INFORMATION_SCHEMA.TABLES WHERE table_schema = 'ANALITICA' AND is_dynamic = 'YES'", 10, '==', '02_dynamic_tables'),
    ('Pipeline', 'Parámetros de la historia', "SELECT COUNT(*) FROM TEC_DEMO.ANALITICA.PARAMETROS", 5, '==', '03b_inteligencia (primera celda)'),
    # Las cifras de la presentación (valores exactos)
    ('Historia', 'Bebidas Bajío oct vs sep (%)', "SELECT ROUND(100 * (SUM(IFF(mes = '2026-10-01', venta_neta, 0)) / SUM(IFF(mes = '2026-09-01', venta_neta, 0)) - 1), 1) FROM TEC_DEMO.ANALITICA.VENTAS_MENSUALES WHERE region = 'Bajío' AND categoria = 'Bebidas'", -31.6, '==', '01 → 02'),
    ('Historia', 'Quiebre Bebidas Bajío oct (%)', "SELECT ROUND(100 * SUM(dias_sku_quiebre) / SUM(dias_sku), 1) FROM TEC_DEMO.ANALITICA.INVENTARIO_MENSUAL WHERE mes = '2026-10-01' AND region = 'Bajío' AND categoria = 'Bebidas'", 34.6, '==', '01 → 02'),
    ('Historia', 'Fill rate Proveedor 07 Bajío oct (%)', "SELECT ROUND(100 * SUM(unidades_recibidas) / SUM(unidades_pedidas), 1) FROM TEC_DEMO.ANALITICA.COMPRAS_MENSUALES WHERE mes = '2026-10-01' AND region = 'Bajío' AND proveedor LIKE 'Proveedor 07%'", 72.6, '==', '01 → 02'),
    ('Historia', 'Primer día con quiebre diario > 20% en León', "SELECT MIN(fecha)::STRING FROM (SELECT fecha FROM TEC_DEMO.RAW.INVENTARIO_DIARIO WHERE region = 'Bajío' AND categoria = 'Bebidas' GROUP BY fecha HAVING COUNT_IF(en_quiebre) / COUNT(*) > 0.2)", '2026-10-01', '==', '01_datos'),
    ('Historia', 'Mapa cuadra con ventas (oct)', "SELECT IFF(ROUND((SELECT SUM(venta_neta) FROM TEC_DEMO.ANALITICA.TIENDAS_GEO WHERE mes = '2026-10-01')) = ROUND((SELECT SUM(venta_neta) FROM TEC_DEMO.ANALITICA.VENTAS_MENSUALES WHERE mes = '2026-10-01')), 1, 0)", 1, '==', '02_dynamic_tables'),
    ('Gobierno', 'Tablas con RAP_REGION', "SELECT COUNT(DISTINCT ref_entity_name) FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES(POLICY_NAME => 'TEC_DEMO.ANALITICA.RAP_REGION'))", 15, '==', '03_gobierno, 03b y 03c'),
    ('Gobierno', 'GERENTE_NORTE no ve llamadas del Bajío', "EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_NORTE') AS SELECT COUNT_IF(region = 'Bajío') FROM TEC_DEMO.ANALITICA.LLAMADAS_360", 0, '==', '03c_call_center (gobierno)'),
    ('Gobierno', 'Columnas con MASK_PII', "SELECT COUNT(*) FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES(POLICY_NAME => 'TEC_DEMO.ANALITICA.MASK_PII'))", 3, '==', '03_gobierno'),
    ('Gobierno', 'GERENTE_NORTE ve solo Norte', "EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_NORTE') AS SELECT LISTAGG(DISTINCT region) FROM TEC_DEMO.ANALITICA.VENTAS_MENSUALES", 'Norte', '==', '03_gobierno'),
    # Enmascaramiento parcial: el gerente ve solo el dominio del correo (****@dominio)
    ('Gobierno', 'GERENTE_NORTE ve email enmascarado', "EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_NORTE') AS SELECT IFF(MAX(email) LIKE '****@%', 1, 0) FROM TEC_DEMO.ANALITICA.VENTAS_ENRIQUECIDAS WHERE venta_id < 1000", 1, '==', '03_gobierno'),
    ('IA', 'Quejas Bajío oct por desabasto', "SELECT COUNT(*) FROM TEC_DEMO.ANALITICA.VOZ_CLIENTE v JOIN TEC_DEMO.ANALITICA.TEMAS_COMENTARIO t ON t.comentario_id = v.comentario_id WHERE v.region = 'Bajío' AND v.mes = '2026-10-01' AND v.calificacion <= 2 AND t.tema = 'Desabasto de producto'", 274, '==', '03b_inteligencia'),
    ('IA', 'Llamadas del Bajío en oct con logística negativa', "SELECT COUNT(*) FROM TEC_DEMO.ANALITICA.LLAMADAS_360 WHERE region = 'Bajío' AND mes = '2026-10-01' AND sentimiento_logistica = 'negative'", 3, '==', '03c_call_center'),
    ('IA', 'Llamadas del Bajío en oct por desabasto', "SELECT COUNT(*) FROM TEC_DEMO.ANALITICA.LLAMADAS_360 WHERE region = 'Bajío' AND mes = '2026-10-01' AND categoria = 'Desabasto de producto'", 2, '==', '03c_call_center'),
    ('IA', 'Venta perdida oct Bajío (M MXN)', "SELECT ROUND((venta_esperada - venta_real) / 1e6, 1) FROM TEC_DEMO.ANALITICA.PRONOSTICO_MENSUAL WHERE region = 'Bajío' AND mes = '2026-10-01'", 8.4, '==', '03b_inteligencia'),
    ('IA', 'Venta en riesgo nov Bajío (M MXN)', "SELECT ROUND((venta_esperada - venta_sin_accion) / 1e6, 1) FROM TEC_DEMO.ANALITICA.PRONOSTICO_MENSUAL WHERE region = 'Bajío' AND mes = '2026-11-01'", 9.1, '==', '03b_inteligencia'),
    ('IA', 'Excedente de Tultitlán para León (u)', "SELECT excedente_origen_unidades FROM TEC_DEMO.ANALITICA.RED_ABASTO WHERE cedis_destino = 'CEDIS León' ORDER BY distancia_km LIMIT 1", 88365, '==', '03b_inteligencia'),
    ('IA', 'CEDIS más cercano a León', "SELECT cedis_origen FROM TEC_DEMO.ANALITICA.RED_ABASTO WHERE cedis_destino = 'CEDIS León' ORDER BY distancia_km LIMIT 1", 'CEDIS Tultitlán', '==', '03b_inteligencia'),
    ('Agente', 'Semantic View', "SELECT COUNT(*) FROM TEC_DEMO.INFORMATION_SCHEMA.SEMANTIC_VIEWS WHERE name = 'SV_COMERCIAL'", 1, '==', '04_semantic_view'),
    ('Agente', 'Servicios de Cortex Search (documentos, feedback y llamadas, con uno por región)', "SELECT COUNT(*) FROM TEC_DEMO.INFORMATION_SCHEMA.CORTEX_SEARCH_SERVICES", 7, '==', '05_search'),
    ('Agente', 'Llamadas en la Semantic View', "SELECT COUNT(*) FROM TEC_DEMO.INFORMATION_SCHEMA.SEMANTIC_TABLES WHERE semantic_view_name = 'SV_COMERCIAL' AND name = 'LLAMADAS'", 1, '==', '04_semantic_view'),
]

# Huella: filas y suma de control de las 18 tablas contra control.csv. Las expresiones viven en
# TEC_DEMO.RAW.HUELLA_CARGA y las evalúa el procedimiento (una sola fuente, no se repiten aquí).
filas = []
try:
    for r in s.sql('CALL TEC_DEMO.RAW.VERIFICAR_HUELLA()').collect():
        ok = r['ESTADO'] == 'PASS'
        filas.append({'estado': r['ESTADO'], 'área': 'Huella', 'check': r['TABLA'], 'valor': f"{r['FILAS']} filas · {r['SUMA_CONTROL']}",
                      'esperado': f"{r['FILAS_ESPERADAS']} filas · {r['SUMA_ESPERADA']}", 'si_falla': '' if ok else 'Corre 01_datos (o 03b/03c/05 si es referencia)'})
except Exception as e:
    filas.append({'estado': 'FAIL', 'área': 'Huella', 'check': 'VERIFICAR_HUELLA', 'valor': f'ERROR: {str(e)[:80]}',
                  'esperado': '18 tablas PASS', 'si_falla': 'Corre 01_datos'})

for area, nombre, sql, esperado, op, si_falla in checks:
    try:
        v = q(sql)
        v_cmp = float(v) if isinstance(esperado, (int, float)) and v is not None else v
        ok = (v_cmp >= esperado) if op == '>=' else (v_cmp == esperado)
    except Exception as e:
        v, ok = f'ERROR: {str(e)[:80]}', False
    filas.append({'estado': 'PASS' if ok else 'FAIL', 'área': area, 'check': nombre, 'valor': str(v),
                  'esperado': f'{op} {esperado}', 'si_falla': '' if ok else f'Corre {si_falla}'})

# Agente publicado y objetos de Telegram (SHOW + RESULT_SCAN)
def show(sql, col='"name"'):
    s.sql(sql).collect()
    return [r[0] for r in s.sql(f'SELECT {col} FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))').collect()]

agentes = show('SHOW AGENTS IN SCHEMA TEC_DEMO.AGENTES')
filas.append({'estado': 'PASS' if 'AGENTE_COMERCIAL' in agentes else 'FAIL', 'área': 'Agente', 'check': 'AGENTE_COMERCIAL existe',
              'valor': ', '.join(agentes), 'esperado': 'AGENTE_COMERCIAL', 'si_falla': '' if 'AGENTE_COMERCIAL' in agentes else 'Corre 06_agente'})
tareas = dict(zip(show('SHOW TASKS IN SCHEMA TEC_DEMO.AGENTES'), show('SHOW TASKS IN SCHEMA TEC_DEMO.AGENTES', '"state"')))
estado_task = tareas.get('RESPONDER_TELEGRAM_TASK', 'no existe')
filas.append({'estado': 'INFO', 'área': 'Telegram', 'check': 'Task bidireccional', 'valor': estado_task,
              'esperado': 'suspended (resume solo en la demo)', 'si_falla': ''})

res = pd.DataFrame(filas)
print(f"{(res.estado == 'PASS').sum()} PASS · {(res.estado == 'FAIL').sum()} FAIL")
for _, r in res[res.estado == 'FAIL'].iterrows():
    print(f"  FAIL · {r['check']}: {r['valor']} ({r['si_falla']})")
res

### Prueba rápida del agente (opcional, ~1 min)
Hace la pregunta principal **como GERENTE_BAJIO**. Si la respuesta menciona *Proveedor 07*, *34.6%* y *Tultitlán*, el agente está listo para el escenario.

In [ ]:
%%sql -r prueba_agente
USE ROLE GERENTE_BAJIO;
WITH r AS (SELECT TRY_PARSE_JSON(SNOWFLAKE.CORTEX.DATA_AGENT_RUN('TEC_DEMO.AGENTES.AGENTE_COMERCIAL',
  $${"messages":[{"role":"user","content":[{"type":"text","text":"¿Por qué cayeron las ventas en el Bajío en octubre de 2026? Dame la causa raíz y 3 acciones."}]}]}$$)) j),
t AS (SELECT LISTAGG(f.value:text::STRING, ' ') AS respuesta FROM r, LATERAL FLATTEN(j:content) f WHERE f.value:type = 'text')
SELECT IFF(CONTAINS(respuesta, 'Proveedor 07') AND CONTAINS(respuesta, '34.6'), 'PASS', 'REVISAR') AS estado,
       CONTAINS(respuesta, 'Tultitlán') AS menciona_cedis, respuesta FROM t;

In [ ]:
%%sql -r ctx_fin
USE ROLE TEC_ADMIN;